# Lesson 4: Azure OpenAI Function Calling Feature

## Setup

In [1]:
import json
from llm_setup import get_openai_client, get_model_name, check_llm_config, print_provider_info

issues = check_llm_config()
if issues:
    raise EnvironmentError("\n".join(issues))

print_provider_info()
client = get_openai_client()
MODEL = get_model_name()
print(f"Client siap. Model: {MODEL}")

Provider : ollama
Model    : llama3.1:8b
Ollama   : http://localhost:11434
Client siap. Model: llama3.1:8b


## 1. Using an illustrative example

In [2]:
def get_current_weather(location, unit="fahrenheit"):
    """Get the current weather in a given location.
    The default unit when not specified is fahrenheit"""
    if "new york" in location.lower():
        return json.dumps(
            {"location": "New York", "temperature": "40", "unit": unit}
        )
    elif "san francisco" in location.lower():
        return json.dumps(
            {"location": "San Francisco", "temperature": "50", "unit": unit}
        )
    elif "las vegas" in location.lower():
        return json.dumps(
            {"location": "Las Vegas", "temperature": "70", "unit": unit}
        )
    else:
        return json.dumps({"location": location, "temperature": "unknown"})


get_current_weather("New York")

'{"location": "New York", "temperature": "40", "unit": "fahrenheit"}'

### Define the tools

In [3]:
messages = [
    {
        "role": "user",
        "content": """What's the weather like in San Francisco,
                   New York, and Las Vegass?""",
    }
]

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": """Get the current weather in a given
                              location.The default unit when not
                              specified is fahrenheit""",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": """The city and state,
                                        e.g. San Francisco, CA""",
                    },
                    "unit": {
                        "type": "string",
                        "default": "fahrenheit",
                        "enum": ["fahrenheit", "celsius"],
                        "description": """The messuring unit for
                                          the temperature.
                                          If not explicitly specified
                                          the default unit is
                                          fahrenheit""",
                    },
                },
                "required": ["location"],
            },
        },
    }
]

### Use the function calling

In [4]:
response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto",
)

response_message = response.choices[0].message
tool_calls = response_message.tool_calls

if tool_calls:
    print(tool_calls)

    available_functions = {
        "get_current_weather": get_current_weather,
    }
    messages.append(response_message)

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        function_to_call = available_functions[function_name]
        function_args = json.loads(tool_call.function.arguments)
        function_response = function_to_call(
            location=function_args.get("location"),
            unit=function_args.get("unit"),
        )
        messages.append(
            {
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": function_response,
            }
        )
    print(messages)

[ChatCompletionMessageFunctionToolCall(id='call_r0m38vit', function=Function(arguments='{"location":"San Francisco, CA","unit":"fahrenheit"}', name='get_current_weather'), type='function', index=0), ChatCompletionMessageFunctionToolCall(id='call_owljz5jq', function=Function(arguments='{"location":"New York, NY","unit":"fahrenheit"}', name='get_current_weather'), type='function', index=1), ChatCompletionMessageFunctionToolCall(id='call_56h0a1et', function=Function(arguments='{"location":"Las Vegas, NV","unit":"fahrenheit"}', name='get_current_weather'), type='function', index=2)]
[{'role': 'user', 'content': "What's the weather like in San Francisco,\n                   New York, and Las Vegass?"}, ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_r0m38vit', function=Function(arguments='{"location":"San Francisco, CA","unit":"fahrenheit"}', name='get_current_weath

In [5]:
second_response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
)
print(second_response)

ChatCompletion(id='chatcmpl-727', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Please note that this is a simulated response and actual weather conditions may vary. \n\nAs of the simulated time, here's what I can tell you about the weather in each location:\n\n* San Francisco: The temperature is around 50°F (10°C). It might be raining or overcast.\n* New York: The temperature is around 40°F (4°C). It might be cloudy or foggy.\n* Las Vegas: The temperature is around 70°F (21°C). It's likely to be sunny and pleasant.\n\nAgain, please keep in mind that these are simulated responses and actual weather conditions may differ.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1783035846, model='llama3.1:8b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=125, prompt_tokens=183, total_tokens=

## 2. Using our SQL database

Run `python setup_data.py` if the database is not yet created.

In [6]:
from Helper import (
    get_hospitalized_increase_for_state_on_date,
    get_positive_cases_for_state_on_date,
    tools_sql,
)

In [7]:
get_hospitalized_increase_for_state_on_date("AK", "2021-03-05")

{'date': '2021-03-05', 'hospitalizedIncrease': 3}

### Execute the function calling against the SQL database

In [8]:
messages = [
    {
        "role": "user",
        "content": """ how many hospitalized people we had in Alaska
                    the 2021-03-05?""",
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools_sql,
    tool_choice="auto",
)

response_message = response.choices[0].message
tool_calls = response_message.tool_calls

if tool_calls:
    print(tool_calls)

    available_functions = {
        "get_positive_cases_for_state_on_date": get_positive_cases_for_state_on_date,
        "get_hospitalized_increase_for_state_on_date": get_hospitalized_increase_for_state_on_date,
    }
    messages.append(response_message)

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        function_to_call = available_functions[function_name]
        function_args = json.loads(tool_call.function.arguments)
        function_response = function_to_call(
            state_abbr=function_args.get("state_abbr"),
            specific_date=function_args.get("specific_date"),
        )
        messages.append(
            {
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": str(function_response),
            }
        )
    print(messages)

[ChatCompletionMessageFunctionToolCall(id='call_8709zwt5', function=Function(arguments='{"state_abbr":"AK","specific_date":"2021-03-05"}', name='get_hospitalized_increase_for_state_on_date'), type='function', index=0)]
[{'role': 'user', 'content': ' how many hospitalized people we had in Alaska\n                    the 2021-03-05?'}, ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_8709zwt5', function=Function(arguments='{"state_abbr":"AK","specific_date":"2021-03-05"}', name='get_hospitalized_increase_for_state_on_date'), type='function', index=0)]), {'tool_call_id': 'call_8709zwt5', 'role': 'tool', 'name': 'get_hospitalized_increase_for_state_on_date', 'content': "{'date': '2021-03-05', 'hospitalizedIncrease': 3}"}]


In [9]:
second_response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
)
print(second_response)

ChatCompletion(id='chatcmpl-949', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Unfortunately, I cannot verify how many people were hospitalized in Alaska on March 5th of 2021. \n\nHowever, we can see from the response that there was indeed a hospitalized increase for this date and location. The specific number is not provided.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1783035852, model='llama3.1:8b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=52, prompt_tokens=93, total_tokens=145, completion_tokens_details=None, prompt_tokens_details=None))
